In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from sklearn.metrics import mean_squared_error,r2_score
import pandas as pnd

<h1 style="color: red;">Section 1: Data</h1>

<h2>1) Préparation de données</h2>

In [2]:
np.random.seed(44) #à chaque exécution,générer le même dataset de manière aléatoire
# Coefficients
a1, a2, b = 2, 3, 5  # y = 2*X1 + 3*X2 + 5 + bruit
nombre_points = 100 # Nombre de points
# Génération des deux features (X1 et X2)
X1 = np.random.rand(nombre_points) * 10
X2 = np.random.rand(nombre_points) * 10
# Empilement des features dans une seule matrice (shape: (100, 2))
X = np.column_stack((X1, X2))
# Génération du bruit
bruit = np.random.randn(nombre_points) * 2  # Bruit
# Calcul de la target
y = a1 * X1 + a2 * X2 + b + bruit

#spilt data
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=23)
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

(80, 2) (20, 2) (80,) (20,)


<h3>1-d) Importance de la division en training set et test set</h3>
<p>Le <b>training set</b> sert à ajuster (apprendre) les paramètres du modèle : c'est sur ces données que la fonction de coût est calculée et que les poids sont mis à jour.</p>
<p>Le <b>test set</b> n'est jamais utilisé pendant l'apprentissage. Il sert uniquement à évaluer la capacité du modèle à généraliser sur des données qu'il n'a jamais vues.</p>
<p>Sans cette séparation, on ne pourrait pas distinguer un modèle qui a simplement mémorisé les données d'entraînement (sur-apprentissage / overfitting) d'un modèle qui a réellement appris la relation sous-jacente entre les features et la target. Comparer les performances sur train et sur test permet donc de détecter l'overfitting et de vérifier la fiabilité du modèle avant de le déployer.</p>

<h1 style="color: red;">Section 2: Neural network avec tensorflow</h1>

In [ ]:
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense

<h2>2) Modèle de réseau de neurones</h2>

<h3>2-a) Architecture de réseau de neurones</h3>
<p><b>2-a-2) Inputs :</b> le dataset possède 2 features (X1 et X2), le réseau a donc 2 inputs (<code>input_shape=(2,)</code>).</p>
<p><b>2-a-3) Neurones de la couche de sortie :</b> la target y est une valeur continue (régression) : il faut donc <b>1 seul neurone</b> en sortie.</p>
<p><b>2-a-4) Fonction d'activation de sortie :</b> puisque y peut prendre n'importe quelle valeur réelle (non bornée), on utilise une activation <b>linéaire (identité)</b> en sortie.</p>

In [ ]:
model_nn = Sequential()
output_layer = Dense(1, input_shape=(2,), activation='linear')
model_nn.add(output_layer)

In [ ]:
model_nn.compile(optimizer='adam', loss='mse', metrics=['mse'])

<p><i>Remarque :</i> la consigne demande 4000 epochs ; on utilise ici 100 epochs (valeur autorisée si les ressources machine sont limitées).</p>

In [ ]:
history = model_nn.fit(X_train, y_train, epochs=100, verbose=0)

<h3>2-e) Que fait la fonction fit ?</h3>
<p><code>fit()</code> entraîne le réseau de neurones : pour chaque epoch, elle effectue une passe avant (forward pass) sur le training set pour calculer les prédictions, calcule la fonction de coût (ici la MSE) entre prédictions et valeurs réelles, puis effectue une rétropropagation (backpropagation) du gradient de cette erreur et met à jour les poids et le biais via l'optimiseur (ici Adam). Cette opération est répétée pendant le nombre d'epochs indiqué, sur l'ensemble du training set.</p>

<h3>2-f) Nombre de paramètres du réseau</h3>
<p>Calcul à la main : nombre de paramètres = (nombre d'inputs × nombre de neurones de sortie) + nombre de biais (1 par neurone de sortie) = (2 × 1) + 1 = <b>3 paramètres</b> (w1, w2, bias).</p>

In [ ]:
model_nn.summary()
print("Nombre total de paramètres :", model_nn.count_params())

<h3>2-g) Paramètres du réseau de neurones</h3>

In [ ]:
W_nn, bias_nn = model_nn.layers[0].get_weights()

print("Poids :", W_nn.flatten())
print("Biais :", bias_nn)

<h3>2-h) Dessiner le réseau de neurones</h3>

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
input_pos = [(0, 1), (0, -1)]
output_pos = (2, 0)
labels = ['X1', 'X2']

for (x, y_), lab in zip(input_pos, labels):
    ax.scatter(x, y_, s=1500, color='#a8d8ff', edgecolor='black', zorder=3)
    ax.text(x, y_, lab, ha='center', va='center', fontsize=12, zorder=4)

ax.scatter(*output_pos, s=1800, color='#ffb3b3', edgecolor='black', zorder=3)
ax.text(*output_pos, 'ŷ', ha='center', va='center', fontsize=13, zorder=4)

for (x, y_), w in zip(input_pos, W_nn.flatten()):
    ax.annotate('', xy=output_pos, xytext=(x, y_), arrowprops=dict(arrowstyle='->', lw=1.5))
    mx, my = (x + output_pos[0]) / 2, (y_ + output_pos[1]) / 2
    ax.text(mx, my, f"w={w:.3f}", fontsize=10, ha='center', backgroundcolor='white')

ax.annotate('', xy=output_pos, xytext=(output_pos[0], output_pos[1] + 1.5), arrowprops=dict(arrowstyle='->', lw=1.5, color='gray'))
ax.text(output_pos[0], output_pos[1] + 1.6, f"bias={bias_nn[0]:.3f}", fontsize=10, ha='center')

ax.set_xlim(-0.5, 3)
ax.set_ylim(-2, 2.5)
ax.axis('off')
ax.set_title("Architecture du réseau de neurones (régression)")
plt.show()

<h3>2-i) Dans quels cas la régularisation (tuning) est-elle importante ?</h3>
<p>La régularisation devient importante lorsque le modèle présente un risque de <b>sur-apprentissage (overfitting)</b>, notamment dans les cas suivants :</p>
<ul>
<li>Le dataset est petit par rapport à la complexité du modèle (peu d'exemples, beaucoup de paramètres).</li>
<li>L'écart entre la performance sur le training set et sur le test set est important (très bon score sur train, mauvais sur test).</li>
<li>Le modèle est complexe (plusieurs couches cachées, beaucoup de neurones) par rapport à la simplicité du problème.</li>
<li>Les données contiennent du bruit important ou des features corrélées/redondantes.</li>
</ul>
<p>Des techniques comme L1/L2 (weight decay), le dropout, l'early stopping ou la réduction de la taille du réseau permettent alors de limiter l'overfitting.</p>

<h2>3) Prédiction en utilisant le modèle</h2>

In [ ]:
yhat_nn = model_nn.predict(X_test)
yhat_nn = yhat_nn.flatten()

<h3>3-b) Prédiction manuelle (sans predict), avec les matrices</h3>
<p>L'activation de sortie étant linéaire, la prédiction se résume à : ŷ = X . W + bias</p>

In [ ]:
yhat_manual = (X_test @ W_nn + bias_nn).flatten()
print("Prédictions identiques (predict vs calcul manuel) :", np.allclose(yhat_manual, yhat_nn, atol=1e-4))

<h2>4) Évaluation du modèle</h2>

<h3>4-a) Performance du modèle sur le training set</h3>

In [ ]:
yhat_train = model_nn.predict(X_train).flatten()
mse_train, r2_train = mean_squared_error(y_train, yhat_train), r2_score(y_train, yhat_train)
print("MSE train :", mse_train, "- R2 train :", r2_train)

residus_train = y_train - yhat_train
plt.scatter(yhat_train, residus_train, color='blue', alpha=0.6)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel("Valeurs prédites (ŷ)")
plt.ylabel("Résidus (y - ŷ)")
plt.title("Graphique des résidus - Training set")
plt.grid(True)
plt.show()

<p><b>Interprétation :</b> si les résidus sont répartis aléatoirement autour de 0, sans structure ni tendance particulière, cela indique que le modèle capture correctement la relation linéaire entre les features et la target, et qu'il n'y a pas de biais systématique dans les prédictions.</p>

<h3>4-b) Performance du modèle sur le test set</h3>

In [ ]:
yhat_nn_test = model_nn.predict(X_test).flatten()
mse_nn, r2_nn = mean_squared_error(y_test, yhat_nn_test), r2_score(y_test, yhat_nn_test)
print("MSE test :", mse_nn, "- R2 test :", r2_nn)

residus_test = y_test - yhat_nn_test
plt.scatter(yhat_nn_test, residus_test, color='blue', alpha=0.6)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel("Valeurs prédites (ŷ)")
plt.ylabel("Résidus (y - ŷ)")
plt.title("Graphique des résidus - Test set")
plt.grid(True)
plt.show()

<h3>4-c) Évaluer sur training set et test set en même temps : à quoi ça sert ?</h3>
<p>Comparer les deux performances permet de diagnostiquer l'état d'apprentissage du modèle :</p>
<ul>
<li>Erreur faible sur train mais élevée sur test → <b>overfitting</b> (le modèle a mémorisé le training set mais généralise mal).</li>
<li>Erreur élevée sur train et sur test → <b>underfitting</b> (le modèle est trop simple ou insuffisamment entraîné).</li>
<li>Erreurs faibles et proches sur train et test → le modèle a correctement appris et généralise bien.</li>
</ul>

<h1>From scratch</h1>

<h1 style="color: red;"> Section 3 :Régression linéaire from scratch </h1>

<h2>Exercice 2 : Régression linéaire from scratch (version matricielle)</h2>
<h3>1) Calculs élémentaires pour une itération</h3>
<p>Dataset jouet :</p>
<table border="1" cellpadding="4">
<tr><th>X1</th><th>X2</th><th>Y</th></tr>
<tr><td>1</td><td>2</td><td>10</td></tr>
<tr><td>3</td><td>4</td><td>20</td></tr>
<tr><td>5</td><td>6</td><td>30</td></tr>
<tr><td>7</td><td>8</td><td>40</td></tr>
</table>
<p>Modèle initial : bias = 0.1, W = [[0.2], [0.3]]</p>

In [3]:
X_toy = np.array([[1, 2], [3, 4], [5, 6], [7, 8]], dtype=float)
Y_toy = np.array([[10], [20], [30], [40]], dtype=float)
bias_toy = 0.1
W_toy = np.array([[0.2], [0.3]])

<h4>a) Calcul de la prédiction</h4><p>Ŷ = X . W + bias</p>

In [4]:
Y_pred_toy = X_toy @ W_toy + bias_toy
print("Y_pred :\n", Y_pred_toy)

Y_pred :
 [[0.9]
 [1.9]
 [2.9]
 [3.9]]


<h4>b) Calcul des erreurs</h4><p>erreur = Ŷ - Y</p>

In [5]:
error_toy = Y_pred_toy - Y_toy
print("erreur :\n", error_toy)

erreur :
 [[ -9.1]
 [-18.1]
 [-27.1]
 [-36.1]]


<h4>c) Calcul des gradients</h4><p>dW = (1/n) . X^T . erreur, db = (1/n) . somme(erreur)</p>

In [6]:
n_toy = len(X_toy)
dW_toy = (1 / n_toy) * (X_toy.T @ error_toy)
db_toy = (1 / n_toy) * np.sum(error_toy)
print("dW :\n", dW_toy)
print("db :", db_toy)

dW :
 [[-112.9]
 [-135.5]]
db : -22.6


<h4>d) Mise à jour du modèle</h4><p>W = W - learning_rate . dW, b = b - learning_rate . db</p>

In [7]:
learning_rate_toy = 0.0001
W_toy_new = W_toy - learning_rate_toy * dW_toy
bias_toy_new = bias_toy - learning_rate_toy * db_toy
print("W après 1 itération :\n", W_toy_new)
print("bias après 1 itération :", bias_toy_new)

W après 1 itération :
 [[0.21129]
 [0.31355]]
bias après 1 itération : 0.10226


<h2>Modèle (version1) de régression linéaire from scratch avec utilisation des matrices</h2>

<h3>2) Entraîner le modèle sur le dataset de la Section 1</h3>

In [8]:
learning_rate = 0.0001
epochs = 1000


# Initialisation des paramètres
W = np.array([[0.0], [0.0]])  # Shape: (2, 1)
b = 0.0

# Reshape y_train pour garantir les dimensions adéquates
y_train = y_train.reshape(-1, 1)  # Shape: (n_samples, 1)
n = len(X_train)

# Entraînement (descente de gradient vectorisée)
for epoch in range(epochs):
    y_pred = X_train @ W + b  # Shape: (n, 1)
    error = y_pred - y_train  # Shape: (n, 1)

    dW = (1 / n) * (X_train.T @ error)  # Shape: (2, 1)
    db = (1 / n) * np.sum(error)        # Scalaire

    W -= learning_rate * dW
    b -= learning_rate * db

print("Paramètres ajustés:")
print(f"W = \n{W}")
print(f"b = {b:.4f}")

Paramètres ajustés:
W = 
[[2.55761346]
 [3.19975219]]
b = 0.5860


<h3>3) Évaluer le modèle</h3>

In [9]:
y_pred_train_scratch = X_train @ W + b
mse_train_scratch = mean_squared_error(y_train, y_pred_train_scratch)
r2_train_scratch = r2_score(y_train, y_pred_train_scratch)

y_pred_test_scratch = X_test @ W + b
mse_test_scratch = mean_squared_error(y_test, y_pred_test_scratch)
r2_test_scratch = r2_score(y_test, y_pred_test_scratch)

print("Train - MSE :", mse_train_scratch, "- R2 :", r2_train_scratch)
print("Test  - MSE :", mse_test_scratch, "- R2 :", r2_test_scratch)

Train - MSE : 7.597365203452631 - R2 : 0.9076849750957134
Test  - MSE : 5.402510380571767 - R2 : 0.9520368718618744


<h3>4) Comparer avec le modèle de régression linéaire de sklearn</h3>

In [10]:
from sklearn.linear_model import LinearRegression

model_sklearn = LinearRegression()
model_sklearn.fit(X_train, y_train)

yhat_sklearn = model_sklearn.predict(X_test)
mse_sklearn = mean_squared_error(y_test, yhat_sklearn)
r2_sklearn = r2_score(y_test, yhat_sklearn)

print("Coefficients sklearn      :", model_sklearn.coef_.ravel(), model_sklearn.intercept_)
print("Coefficients from scratch :", W.ravel(), b)
print()
print("MSE / R2 sklearn      :", mse_sklearn, r2_sklearn)
print("MSE / R2 from scratch :", mse_test_scratch, r2_test_scratch)

Coefficients sklearn      : [1.94893637 3.00146063] [5.22465148]
Coefficients from scratch : [2.55761346 3.19975219] 0.5860413170449176

MSE / R2 sklearn      : 3.559909703919344 0.9683953582202706
MSE / R2 from scratch : 5.402510380571767 0.9520368718618744


<p><b>Comparaison :</b> sklearn résout la régression linéaire par une méthode analytique (moindres carrés), ce qui donne directement les coefficients optimaux. Le modèle from scratch utilise une descente de gradient : avec 1000 epochs et un learning_rate de 0.0001, il s'approche de la solution optimale sans totalement converger. En augmentant le nombre d'epochs et/ou le learning_rate, le modèle from scratch se rapprocherait davantage des coefficients trouvés par sklearn.</p>